Setup and Model Loading

In [2]:
# Install requirements if needed
# pip install torch torchvision opencv-python tqdm

import os
from pathlib import Path
import cv2
import numpy as np
import torch
from tqdm import tqdm
from PIL import Image

# Set up: Must clone ChimpUFE and put yolox_best_only_model.pth in correct path
repo_root = "../Tracking/ChimpUFE"
yolox_weights = f"{repo_root}/assets/weights/yolox_best_only_model.pth"

# Add src/ to sys.path
import sys
sys.path.append(f"{repo_root}/src")
from tracker.yolox.models import YOLOX, YOLOPAFPN, YOLOXHead
from tracker.yolox.utils import fuse_model

# YOLOX config: follow demo_camera_trap.py
def load_yolox(weights_path, device):
    depth, width = 1.33, 1.25
    in_channels = [256, 512, 1024]
    num_classes = 1
    backbone = YOLOPAFPN(depth, width, in_channels=in_channels)
    head = YOLOXHead(num_classes, width, in_channels=in_channels)
    model = YOLOX(backbone, head)
    model.head.initialize_biases(1e-2)
    ckpt = torch.load(weights_path, map_location="cpu")
    model.load_state_dict(ckpt["model"])
    model = model.to(device).eval()
    model = fuse_model(model)
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
yolox_model = load_yolox(yolox_weights, device)

def detect_body_yolox(img, yolox_model, device, input_size=(800, 1440), conf_thres=0.3):
    # Resize, pad like demo_camera_trap
    img_copy = img.copy()
    h0, w0 = img.shape[:2]
    r = min(input_size[0] / h0, input_size[1] / w0)
    new_w, new_h = int(w0 * r), int(h0 * r)
    resized = cv2.resize(img, (new_w, new_h))
    padded = np.ones((input_size[0], input_size[1], 3), dtype=np.uint8) * 114
    padded[:new_h, :new_w, :] = resized
    padded = padded.transpose(2,0,1)[None].astype(np.float32)
    padded = torch.from_numpy(padded).to(device).float()

    with torch.no_grad():
        preds = yolox_model(padded)
        pred = preds[0]
        # Only keep boxes with conf > threshold
        output = []
        for row in pred:
            x_c, y_c, w, h, obj_conf, class_conf = row[:6].cpu().numpy()
            conf = obj_conf * class_conf
            if conf < conf_thres: continue
            # x_c, y_c, w, h are relative to padded image
            x, y = x_c - w/2, y_c - h/2
            x1, y1 = int(x/r), int(y/r)
            x2, y2 = int((x+w)/r), int((y+h)/r)
            output.append([x1, y1, x2, y2, conf])
        # Sort by confidence
        output = sorted(output, key=lambda x: -x[4])
        return output

# INPUT & OUTPUT FOLDERS
chimp_pic_dir = "../../ChimpPic/train"
out_crop_dir = "../../ChimpPic/ChimpPic_face_crops"
os.makedirs(out_crop_dir, exist_ok=True)

for chimp_name in os.listdir(chimp_pic_dir):
    in_chimp_folder = os.path.join(chimp_pic_dir, chimp_name)
    if not os.path.isdir(in_chimp_folder): continue
    out_chimp_folder = os.path.join(out_crop_dir, chimp_name)
    os.makedirs(out_chimp_folder, exist_ok=True)
    for fname in tqdm(os.listdir(in_chimp_folder), desc=chimp_name):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        in_img_path = os.path.join(in_chimp_folder, fname)
        img = cv2.imread(in_img_path)
        if img is None:
            print(f"Could not load {in_img_path}")
            continue
        # Detect the chimp body
        dets = detect_body_yolox(img, yolox_model, device)
        if not dets:
            print(f"No chimp detected in {in_img_path}")
            continue
        x1, y1, x2, y2, conf = dets[0]  # Take most confident
        face_h = int(1 * (y2 - y1))        # plus haut que 0.35
        delta = int(0.05 * (y2 - y1))         # petit décalage vers le bas (ex, pour éviter de trop couper le haut de tête)
        new_y1 = max(0, y1 + delta)
        new_y2 = min(img.shape[0], new_y1 + face_h)
        face_crop = img[new_y1:new_y2, x1:x2]
        out_path = os.path.join(out_chimp_folder, fname)
        cv2.imwrite(out_path, face_crop)

Amadi:  29%|██▊       | 2/7 [00:01<00:03,  1.65it/s]

No chimp detected in ../../ChimpPic/train/Amadi/Amadi1_frame_0356.png


Amadi:  43%|████▎     | 3/7 [00:02<00:02,  1.40it/s]

No chimp detected in ../../ChimpPic/train/Amadi/Amadi1_frame_0237.png


Kassongo:  25%|██▌       | 1/4 [00:00<00:02,  1.17it/s]

...

Kalemi: 0it [00:00, ?it/s]
Ivan:  50%|█████     | 3/6 [00:02<00:02,  1.38it/s]

No chimp detected in ../../ChimpPic/train/Ivan/Ivan1_frame_0380.png


Jeje:  67%|██████▋   | 2/3 [00:01<00:00,  1.33it/s]

No chimp detected in ../../ChimpPic/train/Jeje/Jeje1_frame_0297.png


Banalia: 100%|██████████| 5/5 [00:03<00:00,  1.61it/s]


Inference Helper (face detection)

In [2]:
# Cell 1 - Imports et préparation
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
import os

import sys
sys.path.append("../Tracking/ChimpUFE")  # <-- Change to actual ChimpUFE repo root
from src.face_embedder.vision_transformer import vit_base
# GPU ou CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)

Running on: cuda


NameError: name 'YOLO' is not defined